# Checkpoint Inspection and Inference

Load a trained model - either a training checkpoint (via
`common.ckpt_loading.restore_eval_weights`, which prefers EMA weights) or an
exported SavedModel - run inference on a folder of images, draw box + polygon +
distance overlays, and inspect the raw head statistics (polygon confidence
histogram, per-detection distance distribution).

The model contract: it has **no internal normalisation**. Inputs are float32,
letterboxed to the model size, and scaled to `[0, 1]`. In `deploy=True` mode the
model returns decoded detections:

| key              | shape                | meaning                                   |
|------------------|----------------------|-------------------------------------------|
| `bbox`           | `[B, max, 4]`        | yxyx normalised                           |
| `classes`        | `[B, max]`           | class id                                  |
| `confidence`     | `[B, max]`           | detection score                           |
| `num_detections` | `[B]`                | valid detections per image                |
| `polygons`       | `[B, max, 24, 3]`    | `(conf, dist, angle)` already activated   |
| `distance`       | `[B, max]`           | distance in metres (already exp'd)        |

Set the parameters below.

## Parameters

In [ ]:
# Experiment YAML matching the checkpoint (defines the model architecture).
CONFIG_PATH = "configs/experiments/yolo/yolov8_poly_dist.yaml"

# EITHER a checkpoint prefix (e.g. ".../ckpt-51498") with CONFIG_PATH,
# OR a SavedModel directory. Leave the unused one as None.
CHECKPOINT = None            # e.g. "/tmp/yolo_run/ckpt-51498"
SAVED_MODEL = None           # e.g. "/tmp/yolo_export"

# Folder of images to run inference on (jpg / png).
IMAGE_DIR = "/path/to/images"

# Per-detection score gate for drawing; polygon bins use the 0.4 decode gate.
SCORE_THRESH = 0.30

import os
assert os.path.isdir("models"), "Run from the repo root (directory containing models/, common/, ...)."

In [ ]:
import glob
import math
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from configs.class_map import DETECTION_CLASSES
CLASS_NAMES = [DETECTION_CLASSES[i] for i in sorted(DETECTION_CLASSES)]
print("TensorFlow", tf.__version__)

## Load the model

For a checkpoint we build the architecture from config, set `deploy=True`, and
restore weights with `restore_eval_weights` (EMA-preferring). For a SavedModel we
load its `serving_default` signature: the exported SavedModel is the **device
contract** (flat per-head tensors, raw `[0, 255]` input, no in-graph NMS), so we
reconstruct the deploy dict above from those heads with
`utils.export.device_decode.reconstruct_detections` — the cells below are then
identical for both sources. Both expose a `run(batch)` callable taking a float32
`[B, H, W, 3]` batch in `[0, 1]`.

The device export is often rectangular (e.g. 672×416); this notebook letterboxes
square to `MODEL_SIZE`, so for a rectangular export the checkpoint path is the
simplest inspection route (or export at a square `--input_size`).

In [ ]:
from configs.yaml_loader import load_config

config = load_config(CONFIG_PATH)
MODEL_SIZE = int(config.task.model.input_size[0])

if SAVED_MODEL:
    from utils.export.device_decode import (
        reconstruct_detections, is_device_contract, is_legacy_contract)
    loaded = tf.saved_model.load(SAVED_MODEL)
    infer = loaded.signatures["serving_default"]
    out_keys = list(infer.structured_outputs.keys())
    mh, mw = int(infer.inputs[0].shape[1]), int(infer.inputs[0].shape[2])
    MODEL_SIZE = mh or MODEL_SIZE

    if is_device_contract(out_keys):
        # Device contract: flat per-head tensors, raw [0, 255] input, one image at a
        # time. Reconstruct the deploy dict (boxes + polygons + distance) per image.
        def run(batch):
            imgs = (np.asarray(batch) * 255.0).astype(np.float32)
            per = [reconstruct_detections(dict(infer(input_image=tf.constant(imgs[i:i + 1]))),
                                          mh, mw, legacy_box_order=True)
                   for i in range(imgs.shape[0])]
            return {k: np.concatenate([p[k] for p in per], axis=0) for k in per[0]}
        print("Loaded device-contract SavedModel:", SAVED_MODEL, "| size", f"{mh}x{mw}")
    elif is_legacy_contract(out_keys):
        def run(batch):
            return infer(tf.constant(batch, tf.float32))
        print("Loaded post-processed SavedModel:", SAVED_MODEL, "| size", MODEL_SIZE)
    else:
        raise SystemExit(f"Unknown SavedModel signature outputs: {sorted(out_keys)}")
elif CHECKPOINT:
    from models.yolo_v8 import build_yolov8
    from common.ckpt_loading import restore_eval_weights

    model = build_yolov8(config.task.model)
    model.deploy = True
    model.build_and_init(config.task.model.input_size)
    kind = restore_eval_weights(model, CHECKPOINT)
    print(f"Restored {kind} weights from {CHECKPOINT} | size {MODEL_SIZE}")

    def run(batch):
        return model(tf.constant(batch, tf.float32), training=False)
else:
    raise SystemExit("Set CHECKPOINT (with CONFIG_PATH) or SAVED_MODEL.")

## Preprocess images

Images are letterboxed to `MODEL_SIZE` (the same math as training/eval) and
scaled to `[0, 1]`. `letterbox_resize` needs box/polygon args; we pass empty
tensors since inference has no ground truth.

In [ ]:
from data_pipeline.augmentations import letterbox_resize

def list_images(folder):
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp")
    files = []
    for e in exts:
        files += glob.glob(os.path.join(folder, e))
        files += glob.glob(os.path.join(folder, e.upper()))
    return sorted(files)

def load_letterboxed(path, size):
    raw = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    raw.set_shape([None, None, 3])
    empty_b = tf.zeros([0, 4], tf.float32)
    empty_p = tf.zeros([0, 2], tf.float32)
    img, _, _ = letterbox_resize(raw, empty_b, empty_p, size, size)
    return img.numpy()          # uint8 [size, size, 3]

files = list_images(IMAGE_DIR)
print(f"{len(files)} image(s) in {IMAGE_DIR}")
assert files, "No images found - check IMAGE_DIR."

MAX_IMAGES = 8
files = files[:MAX_IMAGES]
batch_u8 = np.stack([load_letterboxed(f, MODEL_SIZE) for f in files], axis=0)
batch = batch_u8.astype(np.float32) / 255.0
print("batch:", batch.shape, batch.dtype)

In [ ]:
preds = run(batch)
# SavedModel signatures return plain tensors; unify to numpy dict.
pred = {k: (v.numpy() if hasattr(v, "numpy") else np.asarray(v)) for k, v in preds.items()}
for k, v in pred.items():
    print(f"{k:16s} shape={tuple(v.shape)} dtype={v.dtype}")

## Draw overlays

`common/viz_utils.render_summary_images` draws boxes (with class + score),
radial polygons (conf-gated at 0.4), reusing the exact decode the trainer uses.
Distance is annotated per detection in metres.

In [ ]:
from common.viz_utils import render_summary_images

n = batch.shape[0]
preds_list = []
for i in range(n):
    nd = int(pred["num_detections"][i])
    keep = [j for j in range(nd) if float(pred["confidence"][i, j]) >= SCORE_THRESH]
    preds_list.append({
        "bbox":           pred["bbox"][i],
        "classes":        pred["classes"][i],
        "confidence":     pred["confidence"][i],
        "num_detections": len(keep) and (max(keep) + 1) or 0,
        "polygons":       pred["polygons"][i],
    })

canvas = render_summary_images(
    [batch[i] for i in range(n)], preds_list,
    draw_box=True, draw_poly=True, class_names=CLASS_NAMES, conf_thresh=0.4,
)

cols = 2
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(14, 6 * rows))
for i, ax in enumerate(np.array(axes).reshape(-1)):
    if i < n and canvas is not None:
        ax.imshow(canvas[i])
        # Annotate distances for kept detections.
        nd = int(pred["num_detections"][i])
        for j in range(nd):
            if float(pred["confidence"][i, j]) < SCORE_THRESH:
                continue
            y1, x1, y2, x2 = pred["bbox"][i, j]
            d = float(pred["distance"][i, j])
            ax.text(x1 * MODEL_SIZE, y2 * MODEL_SIZE + 12, f"{d:.1f} m",
                    color="yellow", fontsize=8,
                    bbox=dict(facecolor="black", alpha=0.5, pad=1))
        ax.set_title(os.path.basename(files[i]), fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## Raw head statistics

Two diagnostics aggregated over all kept detections in the batch:

- **Polygon confidence histogram** - `polygons[..., 0]` (already sigmoid). A
  healthy head is bimodal (occupied bins near 1, empty near 0); mass piling above
  the 0.4 gate on empty bins produces spiky contour artifacts.
- **Distance distribution** - `distance` (metres). The trained valid range is
  `[0.5, 10.0]` m.

In [ ]:
conf_vals = []
dist_vals = []
for i in range(n):
    nd = int(pred["num_detections"][i])
    for j in range(nd):
        if float(pred["confidence"][i, j]) < SCORE_THRESH:
            continue
        conf_vals.append(pred["polygons"][i, j, :, 0])   # [24] per-bin conf
        dist_vals.append(float(pred["distance"][i, j]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
if conf_vals:
    ax1.hist(np.concatenate(conf_vals), bins=40, range=(0, 1), color="steelblue")
    ax1.axvline(0.4, color="red", linestyle="--", label="0.4 decode gate")
    ax1.set_title("Polygon per-bin confidence"); ax1.set_xlabel("sigmoid conf")
    ax1.legend()
else:
    ax1.text(0.5, 0.5, "no detections above SCORE_THRESH", ha="center")

if dist_vals:
    ax2.hist(dist_vals, bins=30, color="seagreen")
    ax2.axvspan(0.5, 10.0, color="orange", alpha=0.15, label="trained range")
    ax2.set_title("Per-detection distance (m)"); ax2.set_xlabel("metres")
    ax2.legend()
else:
    ax2.text(0.5, 0.5, "no detections above SCORE_THRESH", ha="center")
plt.tight_layout(); plt.show()

print(f"detections kept: {len(dist_vals)}")
if dist_vals:
    dv = np.array(dist_vals)
    print(f"distance: min={dv.min():.2f} max={dv.max():.2f} mean={dv.mean():.2f} m")